# 🧪 Exploration des données de Sécurité (Couche Bronze)

Ce notebook permet de visualiser et d'explorer les données de délinquance ingérées dans la couche Bronze.
**Objectif :** Vérifier la qualité des données, identifier les indicateurs disponibles et valider la couverture géographique par arrondissement.

In [ ]:
import os
from src.common.spark_session_manager import get_spark_session
from src.config import SECURITE_BRONZE_PATH
from pyspark.sql.functions import col

# 1. Initialiser la session Spark
spark = get_spark_session(app_name="Exploration_Securite_Bronze")

# 2. Configuration "Look Databricks" (Eager Evaluation)
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 20)

## 📊 Exploration de la Sécurité (Base Communale)
Lecture des données depuis le format **Parquet** (stockage optimisé).

In [ ]:
securite_path = os.path.join(SECURITE_BRONZE_PATH, "security-filtered")
df_securite = spark.read.parquet(securite_path)

# Aperçu des données
df_securite

## 🔬 Analyse Technique des Schémas (Data Profiling)
Vérification des types détectés. Contrairement aux présidentielles, ici presque tout est en String (même les nombres) car nous n'avons pas encore fait le casting en Silver.

In [ ]:
print("--- Analyse des types de données ---")
df_securite.printSchema()

## 🔍 Inventaire des Indicateurs de Sécurité
Quels sont les types de crimes et délits recensés dans notre dataset ?

In [ ]:
df_securite.select("indicateur").distinct().sort("indicateur")

## 🌍 Validation du Périmètre Géographique (Arrondissements)
On s'assure que nous avons bien les 9 arrondissements de Lyon (69381 à 69389).

In [ ]:
df_securite.groupBy("CODGEO_2025").count().sort("CODGEO_2025")

## 📅 Couverture Temporelle
Vérification des années disponibles pour notre futur modèle.

In [ ]:
df_securite.select("annee").distinct().sort("annee")